In [ ]:
import sympy as sp
import numpy as np
from scipy.special import jv
from scipy.sparse import lil_matrix
import matplotlib.pyplot as plt
from scipy.linalg import block_diag
from scipy.optimize import root
import sys
import os
from pathlib import Path

plot_dir = Path("/Users/tmalas/Desktop/Dev/Simulations_local/Code_python")
sys.path.insert(0, str(plot_dir))

from Plot_setting import *

In [ ]:

h_bar=1
e=1
phi_quantum=1/2

#### Caracterization of the circuit #####

w_a = 25.338776456203686
w_b = 2*w_a

phi_a = 0.11
Z_a = 2*phi_quantum**2*phi_a**2
C_a_inv = Z_a*w_a
E_Ca = C_a_inv/2

phi_b = 0.204
Z_b = 2*phi_quantum**2*phi_b**2
C_b_inv = Z_b*w_b
E_Cb = C_b_inv/2

E_J = 37.12*2*np.pi*1.5

#Environment
kappa_b=1/10.4; # 1/ns from Leon's paper

w_d=w_b # Drive frequency
w_p=2*w_a-w_d # Pump Frequency

epsilon_p = 0.5
g2 = jv(1,epsilon_p)*E_J*phi_a**2*phi_b # (Leon's paper)
epsilon_d = 10.12*g2#0.75*g2_max

beta_a_p, beta_b_p = 2*E_J*np.sin(epsilon_p)*phi_a**2/(w_a), 2*E_J*np.sin(epsilon_p)*phi_b**2/(w_b)
B = np.array([beta_a_p, beta_b_p])  # shape (2,)
eps_d_p = epsilon_d*phi_b/w_b
kappa_b_p = kappa_b / w_b

# Cat-state values
b_0_cat_state = 0.0

a_0_cat_state_1 = 1j * np.sqrt(eps_d_p/ beta_b_p)

# Functions

In [ ]:
def build_L(M, C, K, N_H, omega=1.0, nu=1.0):
    """
    Build linear HB operator L for ordering:
    [c0, s1, c1, s2, c2, ..., s_NH, c_NH] with each block in R^n.
    Returns L of shape (n*(2N_H+1), n*(2N_H+1)).
    """
    n = M.shape[0]
    dim = n * (2*N_H + 1)
    L = np.zeros((dim, dim), dtype=float)

    # helper to locate block indices in z
    def idx_c0():      return 0
    def idx_sk(k):     return 1 + 2*(k-1)
    def idx_ck(k):     return 2 + 2*(k-1)

    # DC: (K) acting on c0 (with your 1/sqrt2 convention, L is still K here
    # because the same scaling appears in Gamma and Gamma^+; keep L=K)
    i0 = idx_c0()
    L[i0*n:(i0+1)*n, i0*n:(i0+1)*n] = K

    # Harmonics
    for k in range(1, N_H+1):
        A = K - ((k*omega/nu)**2)*M  # n×n
        B = (k*omega/nu) * C          # n×n

        is_ = idx_sk(k)
        ic_ = idx_ck(k)

        # sine row blocks
        L[is_*n:(is_+1)*n, is_*n:(is_+1)*n] = A
        L[is_*n:(is_+1)*n, ic_*n:(ic_+1)*n] = -B

        # cosine row blocks
        L[ic_*n:(ic_+1)*n, is_*n:(is_+1)*n] = B
        L[ic_*n:(ic_+1)*n, ic_*n:(ic_+1)*n] = A

    return L

def build_b_ext(N_H, eps_d_p, nu = 2):
    """
    b_ext in the same ordering as z:
    [c0, s1, c1, s2, c2, ..., s_NH, c_NH], each block in R^2.
    Forcing: f_ext = -sqrt(2)*eps_d' * cos(2 tau) * e2.
    """
    n = 2
    b = np.zeros(n*(2*N_H+1), dtype=float)

    def idx_ck(k): return 2 + 2*(k-1)

    if N_H >= nu:
        ic2 = idx_ck(nu)
        # e2 = [0,1]
        b[ic2*n:(ic2+1)*n] = np.array([0.0, -np.sqrt(2.0)*eps_d_p])

    return b

def build_Gamma(t, N_H, n, omega=2.0, nu=2.0):
    """
    Build Gamma = I_n ⊗ Phi with omega/nu = 1.

    t   : array of shape (N,)
    N_H : number of harmonics
    n   : number of DOFs
    """
    t = np.asarray(t)
    N = t.size
    k = np.arange(1, N_H + 1)

    # N x N_H
    S = np.sin(np.outer(t, k*omega/nu))
    C = np.cos(np.outer(t, k*omega/nu))

    # Build Phi: N x (2*N_H + 1) with columns [1/sqrt2, sin(1t), cos(1t), sin(2t), cos(2t), ...]
    Phi = np.empty((N, 2 * N_H + 1), dtype=float)
    Phi[:, 0] = 1.0 / np.sqrt(2.0)
    Phi[:, 1::2] = S
    Phi[:, 2::2] = C

    # Gamma: (n*N) x (n*(2*N_H+1))
    Gamma = np.kron(Phi, np.eye(n))
    return Gamma

def x_tilde_to_X(tildex, n, N):
    # tildex is time-major: [x(t1); x(t2); ...], each x(tj) is length n
    return tildex.reshape(N, n).T   # -> (n, N)

def X_to_x_tilde(X):
    # X is (n, N); return time-major vector
    return X.T.reshape(-1)

def f_nl_time(X):
    """
    X: shape (n, N) with n=2
    returns Fnl: shape (n, N)
    """
    y = np.sqrt(2.0) * (X[0, :] + X[1, :])          # shape (N,)
    g = np.cos(y) - 1.0                             # shape (N,)
    Fnl = -np.sqrt(2.0) * B[:, None] * g[None, :]    # (2,N)
    return Fnl

def b_nl(z, Gamma, Gamma_pinv, n, N):
    x_tilde = Gamma @ z
    X = x_tilde_to_X(x_tilde, n, N)
    Fnl = f_nl_time(X)              # (n,N)
    f_tilde = X_to_x_tilde(Fnl)     # (nN,)
    return Gamma_pinv @ f_tilde     # (n*(2N_H+1),)

def residual(z, L, b_ext, Gamma, Gamma_pinv, n, N):
    return L @ z - b_nl(z, Gamma, Gamma_pinv, n, N) - b_ext

def dfnl_dX_blocks(X):
    """
    returns list of N blocks J_j = ∂f_nl/∂X at each time sample
    each block is 2x2
    """
    y = np.sqrt(2.0) * (X[0, :] + X[1, :])
    s = np.sin(y)  # shape (N,)

    # J_j = 2 sin(y_j) * [[beta_a, beta_a],[beta_b, beta_b]]
    base = np.array([[beta_a_p, beta_a_p],
                     [beta_b_p, beta_b_p]])  # 2x2

    J_blocks = [(2.0 * s[j]) * base for j in range(X.shape[1])]
    return J_blocks

def build_dftilde_dx_tilde(X, n, N):
    """
    Block diagonal (nN x nN) with n×n blocks per time sample,
    consistent with Gamma = kron(Phi, I_n) (time-major stacking):
      tilde x = [x(t1); x(t2); ...], each x(tj) in R^n.
    """
    J_blocks = dfnl_dX_blocks(X)  # length N, each (n,n)
    Jbig = lil_matrix((n*N, n*N), dtype=float)

    for j in range(N):
            Jj = J_blocks[j]
            rows = slice(j*n, (j+1)*n)
            cols = slice(j*n, (j+1)*n)
            Jbig[rows, cols] = Jj

    return Jbig.tocsr()

def db_dz(z, Gamma, Gamma_pinv, n, N):
    # time samples
    x_tilde = Gamma @ z
    X = x_tilde_to_X(x_tilde, n, N)

    # build ∂f/∂x in time
    Jbig = build_dftilde_dx_tilde(X, n, N)

    # chain rule
    return (Gamma_pinv @ (Jbig @ Gamma))

def jacobian(z, L, Gamma, Gamma_pinv, n, N):
    return L - db_dz(z, Gamma, Gamma_pinv, n, N)
def newton_solve(z0, L, b_ext, Gamma, Gamma_pinv, n, N, tol=1e-10, maxit=50):
    z = z0.copy()
    for it in range(maxit):
        r = residual(z, L, b_ext, Gamma, Gamma_pinv, n, N)
        norm_r = np.linalg.norm(r)
        if norm_r < tol:
            print("Converged", it, norm_r)
            return z

        J = jacobian(z, L, Gamma, Gamma_pinv, n, N)
        dz = np.linalg.solve(J, -r)

        # optional damping
        z = z + dz
        print(it, norm_r, np.linalg.norm(dz))
    raise RuntimeError("Newton did not converge")

def powell_hybrid_solve(z0, L, b_ext, Gamma, Gamma_pinv, n, N,
                        tol=1e-10, maxit=50, use_jac=True, verbose = True):
    """
    Powell hybrid method (MINPACK hybr) via scipy.optimize.root.
    """

    def fun(z):
        return residual(z, L, b_ext, Gamma, Gamma_pinv, n, N)

    jac = (lambda z: jacobian(z, L, Gamma, Gamma_pinv, n, N)) if use_jac else None

    sol = root(
        fun, z0,
        jac=jac,
        method="hybr",
        tol=tol,
        options={"maxfev": maxit * (len(z0) + 1)}
    )

    if not sol.success:
        raise RuntimeError(f"Powell hybrid did not converge: {sol.message}")

    if verbose : print("Converged", sol.nfev, np.linalg.norm(sol.fun))
    return sol.x
def plot_phase_space(z_solution, N_H, traj=False):
    """
    z_solution: array of length 2*(2*N_H+1)
    """

    # time grid (one period)
    t = np.linspace(0, 2*np.pi, 400)

    # split DOFs
    za = z_solution[::2]        # memory
    zb = z_solution[1::2]        # buffer

    # initialize
    x_a = np.zeros_like(t)
    p_a = np.zeros_like(t)
    x_b = np.zeros_like(t)
    p_b = np.zeros_like(t)

    # DC terms
    x_a += za[0] / np.sqrt(2)
    x_b += zb[0] / np.sqrt(2)

    # harmonics
    for k in range(1, N_H + 1):
        s_idx = 1 + 2*(k-1)
        c_idx = 2 + 2*(k-1)

        # memory (a)
        x_a += za[s_idx]*np.sin(k*t) + za[c_idx]*np.cos(k*t)
        p_a += k*za[s_idx]*np.cos(k*t) - k*za[c_idx]*np.sin(k*t)

        # buffer (b)
        x_b += zb[s_idx]*np.sin(k*t) + zb[c_idx]*np.cos(k*t)
        p_b += k/2*zb[s_idx]*np.cos(k*t) - k/2*zb[c_idx]*np.sin(k*t)

    a = (x_a + 1j*p_a)/np.sqrt(2)
    a_R = a*np.exp(1j*t)*1j

    b = (x_b + 1j*p_b)/np.sqrt(2)
    b_R = b*np.exp(2j*t)

    # ---- plots ----
    plt.figure()
    plt.plot(np.real(a_R), np.imag(a_R))
    plt.xlabel(r"$\Re(a_R)$")
    plt.ylabel(r"$\Im(a_R)$")
    plt.title("Phase portrait: memory (a)")
    plt.grid(True)
    plt.gca().set_aspect('equal')

    plt.figure()
    plt.plot(np.real(b_R), np.imag(b_R))
    plt.xlabel(r"$\Re(b_R)$")
    plt.ylabel(r"$\Im(b_R)$")
    plt.title("Phase portrait: buffer (b)")
    plt.grid(True)
    plt.gca().set_aspect('equal')
    plt.show()
    if traj : return(a_R, b_R)

# Find cat-state and unstable orbit

In [ ]:
n   = 2 # number of modes
N_H = 5 # number of harmonics
# Number of instant for AFT
N = 900*N_H
# Uniform grid
omega = 2.0
nu = 2
tau_j = np.linspace(0, 100*np.pi, N)

z_0_init = [0.0, 0.0,
        np.sqrt(2)*np.imag(a_0_cat_state_1), -0.1, 
        np.sqrt(2)*np.real(a_0_cat_state_1), -0.1] # cat state

z_0 = np.pad(z_0_init, (0,n*(2*N_H+1)-len(z_0_init)), constant_values=0.0)

M = np.array([[1.0, 0.0],
              [0.0, 0.25]])
C = np.array([[0.0, 0.0],
              [0.0, 0.5*kappa_b_p]]) 
K = np.eye(2)
L = build_L(M, C, K, N_H, omega, nu)
Gamma = build_Gamma(tau_j, N_H, n, omega, nu)
Gamma_inverse = np.linalg.pinv(Gamma)
b_ext = build_b_ext(N_H, eps_d_p, nu)
z_sol_cat = powell_hybrid_solve(z_0, L, b_ext, Gamma, Gamma_inverse, n, N)
plot_phase_space(z_sol_cat, N_H)

In [ ]:
n   = 2 # number of modes
N_H = 5 # number of harmonics
# Number of instant for AFT
N = 900*N_H
# Uniform grid
omega = 2.0
nu = 2
tau_j = np.linspace(0, 100*np.pi, N)

z_0_init = np.array([ 2.63294857e-02,  4.52780115e-02, -2.19460257e-07, -2.57413833e-17,
         5.30791448e-06,  1.97743361e-17,  1.32143467e-04, -1.60585607e-02,
        -2.96562402e-03,  3.59303044e-01,  5.26976746e-09,  5.74141999e-08,
         2.33034913e-08,  2.56606252e-07,  1.03811269e-04,  9.05211755e-04,
        -1.15922349e-03, -9.96626089e-03]) #unstable manifold

z_0 = np.pad(z_0_init, (0,n*(2*N_H+1)-len(z_0_init)), constant_values=0.0)

M = np.array([[1.0, 0.0],
              [0.0, 0.25]])
C = np.array([[0.0, 0.0],
              [0.0, 0.5*kappa_b_p]]) 
K = np.eye(2)
L = build_L(M, C, K, N_H, omega, nu)
Gamma = build_Gamma(tau_j, N_H, n, omega, nu)
Gamma_inverse = np.linalg.pinv(Gamma)
b_ext = build_b_ext(N_H, eps_d_p, nu)
z_sol_u = powell_hybrid_solve(z_0, L, b_ext, Gamma, Gamma_inverse, n, N)
a_R_u, b_R_u = plot_phase_space(z_sol_u, N_H, traj = True)
y_u = np.vstack((a_R_u*np.exp(-0.05j), b_R_u*np.exp(-0.1j)))

## We define the cat-state orbit as a function of $\theta_0$

In [ ]:
def orbit_state(z_sol, N_H, tau, theta_0):
    """
    Return the periodic-orbit state y(t) starting from phase theta0.
    This must return the full 4D state along the limit cycle.
    """
    za = z_sol[::2]        # memory
    zb = z_sol[1::2]        # buffer

    # initialize
    x_a = 0
    v_a = 0
    x_b = 0
    v_b = 0

    # DC terms
    x_a += za[0] / np.sqrt(2)
    x_b += zb[0] / np.sqrt(2)
    for k in range(1, N_H + 1):
        s_idx = 1 + 2*(k-1)
        c_idx = 2 + 2*(k-1)

        # memory (a)
        x_a += za[s_idx]*np.sin(k*(tau-theta_0)) + za[c_idx]*np.cos(k*(tau-theta_0))
        v_a += k*za[s_idx]*np.cos(k*(tau-theta_0)) - k*za[c_idx]*np.sin(k*(tau-theta_0))

        # buffer (b)
        x_b += zb[s_idx]*np.sin(k*(tau-theta_0)) + zb[c_idx]*np.cos(k*(tau-theta_0))
        v_b += k*zb[s_idx]*np.cos(k*(tau-theta_0)) - k*zb[c_idx]*np.sin(k*(tau-theta_0))

    return(x_a, x_b, v_a, v_b)

In [ ]:
tau = 14.5*np.pi
theta_0 = 0.1*np.pi
X_0 = orbit_state(z_sol_cat, N_H, tau, theta_0)
X_0

## We define the Jacobian of the extended system, the monodromy matrix and the unstable eigenspace

In [ ]:
epsilon = 0.5
def J_extended(X):
    x_a, x_b = X[:2]
    coupling = np.sin(np.sqrt(2)*x_a + np.sqrt(2)*x_b)
    J_K_nl = np.array([[1+2*beta_a_p*coupling, 2*beta_a_p*coupling],[8*beta_b_p*coupling, 1+8*beta_b_p*coupling]])
    delta_c = np.array([[0.0, 0.0],
              [0.0, 2*kappa_b_p]])
    zeros_mat = np.zeros((n,n)) 
    eye_mat = np.eye(n)
    jacobian = np.block([
        [zeros_mat, eye_mat, zeros_mat, zeros_mat],
        [-J_K_nl, -delta_c, zeros_mat, 2*epsilon*eye_mat],
        [zeros_mat, zeros_mat, zeros_mat, J_K_nl.T],
        [zeros_mat, zeros_mat, -eye_mat, delta_c]
    ])
    return(jacobian)

In [ ]:
J_ext = J_extended(X_0)
J_ext

In [ ]:
from numpy.linalg import eig
from scipy.integrate import solve_ivp

def monodromy_matrix(theta0, T, orbit_state, J_extended, dim=8, rtol=1e-10, atol=1e-12):
    """
    Compute monodromy matrix M for the periodic orbit starting at phase theta0.
    orbit_state(z_sol, N_H, tau, theta0) -> y(tau) on the orbit (shape (dim,))
    jacobian(y) -> J(y) (shape (dim, dim))
    """
    # Variational equation for flattened Psi
    def dPsi_dt(tau, Psi_flat):
        y = orbit_state(z_sol_cat, N_H, tau, theta0)          # state on the orbit at time t
        J = J_extended(y)                     # Jacobian along the orbit
        Psi = Psi_flat.reshape(dim, dim)
        dPsi = J @ Psi
        return dPsi.reshape(dim * dim)

    Psi0 = np.eye(dim).reshape(dim * dim)

    sol = solve_ivp(
        dPsi_dt, (0.0, T), Psi0,
        method="DOP853", rtol=rtol, atol=atol
    )
    PsiT = sol.y[:, -1].reshape(dim, dim)
    return PsiT

def unstable_subspace(M, n_unstable=None, tol=1e-8):
    """
    Return unstable eigenvalues and eigenvectors of M.
    Unstable means |mu| > 1 + tol.
    """
    mu, vecs = eig(M)  # vecs columns are right eigenvectors
    idx = np.where(np.abs(mu) > 1.0 + tol)[0]

    # Optional: enforce expected number of unstable directions
    if n_unstable is not None:
        # choose the n_unstable eigenvalues with largest |mu|
        idx = idx[np.argsort(np.abs(mu[idx]))[::-1][:n_unstable]]

    mu_u = mu[idx]
    Phi_u = vecs[:, idx]  # dim x n_unstable
    return mu_u, Phi_u

def Zq_Zp_from_theta(theta0, T, orbit_state, J_extended, n_unstable=4):
    """
    Compute Z_q and Z_p from the unstable eigenvectors of the monodromy matrix.
    Assumes the state is ordered as (q, p) with q dimension 4 and p dimension 4.
    """
    M = monodromy_matrix(theta0, T, orbit_state, J_extended, dim=8)
    mu_u, Phi_u = unstable_subspace(M, n_unstable=n_unstable)

    # If eigenvectors are complex (can happen), you may prefer real basis
    # If mu comes in complex conjugate pairs, take real/imag parts:
    if np.max(np.abs(np.imag(Phi_u))) > 1e-10:
        # build a real basis from real/imag parts
        real_basis = []
        for k in range(Phi_u.shape[1]):
            v = Phi_u[:, k]
            if np.max(np.abs(np.imag(v))) > 1e-10:
                real_basis.append(np.real(v))
                real_basis.append(np.imag(v))
            else:
                real_basis.append(np.real(v))
        Phi_u = np.stack(real_basis, axis=1)
        # trim to desired dimension
        Phi_u = Phi_u[:, :n_unstable]

    Zq = Phi_u[0:4, :]   # 4 x n_unstable
    Zp = Phi_u[4:8, :]   # 4 x n_unstable
    return M, mu_u, Zq, Zp

In [ ]:
T = 2*np.pi  # example period
M, mu_u, Zq, Zp = Zq_Zp_from_theta(theta_0, T, orbit_state, J_extended, n_unstable=4)
print("unstable multipliers:", mu_u)
print("Zq shape:", Zq.shape, "Zp shape:", Zp.shape)

# Let's sample to get a set of initial values

In [ ]:
def sample_S3(gamma, n_samples):
    """
    Uniform samples on S^3 of radius gamma.
    Returns array shape (n_samples, 4)
    """
    v = np.random.randn(n_samples, 4)
    v /= np.linalg.norm(v, axis=1)[:, None]
    return gamma * v

In [ ]:
n_theta = 10
n_q = 1
gamma = 1e-3
#theta_vals = np.linspace(3*T/32, 5*T/32, n_theta, endpoint=False)
theta_vals = np.linspace(0, T, n_theta, endpoint=False)
#theta_vals = np.linspace(0.6, 0.8, n_theta, endpoint=False)

W_i = {}

for theta_0 in theta_vals:
    q_vals = sample_S3(gamma, n_q)

    # Base state at theta_0
    x_a, x_b, v_a, v_b = orbit_state(z_sol_cat, N_H, theta_0, 0)
    A_0_theta_0 = np.array([v_a,x_b,x_a,v_b])

    # Jacobian blocks / (un)stable frame info at theta_0
    _, _, Zq_theta_0, Zp_theta_0 = Zq_Zp_from_theta(
        theta_0, T, orbit_state, J_extended, n_unstable=4
    )

    Zq_theta_0_inv = np.linalg.inv(Zq_theta_0)

    Wi_theta = []
    for q_val_i in q_vals:
        q = np.asarray(q_val_i).reshape(-1)  # ensure 1D vector

        # First block: perturbed state (requires A_0_theta_0 and q to be same shape)
        x = np.asarray(A_0_theta_0).reshape(-1) + q

        # Second block: mapped vector
        y = np.zeros((4))#(Zp_theta_0 @ Zq_theta_0_inv @ q.reshape(-1, 1)).reshape(-1)

        Wi_theta.append(np.concatenate((x,[theta_0], y)))

    W_i[theta_0] = Wi_theta

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
ax_a, ax_b = axes
for theta_0 in theta_vals:
    x_a, v_a = W_i[theta_0][0][0:4:2]
    x_b, v_b = W_i[theta_0][0][1:4:2]
    a = (x_a + 1j*v_a)/np.sqrt(2)
    b = (x_b + 1j*v_b/2)/np.sqrt(2)

    a_R = a*np.exp(1j*theta_0)
    b_R = b*np.exp(2j*theta_0)

    ax_a.scatter(np.real(a_R), np.imag(a_R))
    ax_b.scatter(np.real(b_R), np.imag(b_R))
ax_a.set_title("Phase portrait: mode a")
ax_a.set_xlabel(r"$\Re(a_R)$")
ax_a.set_ylabel(r"$\Im(a_R)$")
ax_a.set_aspect('equal')
ax_a.grid(True)

ax_b.set_title("Phase portrait: mode b")
ax_b.set_xlabel(r"$\Re(b_R)$")
ax_b.set_ylabel(r"$\Im(b_R)$")
ax_b.set_aspect('equal')
ax_b.grid(True)
plt.show()

# Equations of motion

In [ ]:
import numpy as np

SQRT2 = np.sqrt(2.0)

def propagate_step(t, u, params):
    # u length 10: [x_a,x_b,v_a,v_b,theta,p_x_a,p_x_b,p_v_a,p_v_b,p_theta]
    x_a, x_b, v_a, v_b, theta, p_x_a, p_x_b, p_v_a, p_v_b, p_theta = u
    delta_ca, delta_cb, epsilon_d_p, beta_a_p, beta_b_p, epsilon = params

    s = SQRT2 * (x_a + x_b)
    cos_s = np.cos(s)
    sin_s = np.sin(s)

    # nonlinear forces
    f_nl_a = SQRT2 * beta_a_p * (cos_s - 1.0)
    f_nl_b = SQRT2 * (4.0 * beta_b_p) * (cos_s - 1.0)

    # drive on b
    drive_b = 4.0 * SQRT2 * epsilon_d_p * np.cos(2.0 * theta)

    du = np.empty_like(u)

    # state
    du[0] = v_a
    du[1] = v_b
    du[2] = -delta_ca * v_a - x_a + f_nl_a + 2.0 * epsilon * p_v_a
    du[3] = -delta_cb * v_b - 4.0 * x_b + f_nl_b - drive_b + 2.0 * epsilon * p_v_b
    du[4] = 1.0

    # costates
    # p_xdot = -∂H/∂x : includes BOTH p_v_a and p_v_b contributions
    du[5] = (p_v_a
             + 2.0 * beta_a_p * p_v_a * sin_s
             + 8.0 * beta_b_p * p_v_b * sin_s)

    du[6] = (4.0 * p_v_b
             + 2.0 * beta_a_p * p_v_a * sin_s
             + 8.0 * beta_b_p * p_v_b * sin_s)

    # p_vdot = -∂H/∂v
    du[7] = -p_x_a + delta_ca * p_v_a
    du[8] = -p_x_b + delta_cb * p_v_b

    # p_theta dot = -∂H/∂theta
    du[9] = -8.0 * SQRT2 * epsilon_d_p * p_v_b * np.sin(2.0 * theta)

    return du
def propagate_step_no_p_theta(t, u, params):
    # u length 10: [x_a,x_b,v_a,v_b,theta,p_x_a,p_x_b,p_v_a,p_v_b,p_theta]
    x_a, x_b, v_a, v_b, theta, p_x_a, p_x_b, p_v_a, p_v_b = u
    delta_ca, delta_cb, epsilon_d_p, beta_a_p, beta_b_p, epsilon = params

    s = SQRT2 * (x_a + x_b)
    cos_s = np.cos(s)
    sin_s = np.sin(s)

    # nonlinear forces
    f_nl_a = SQRT2 * beta_a_p * (cos_s - 1.0)
    f_nl_b = SQRT2 * (4.0 * beta_b_p) * (cos_s - 1.0)

    # drive on b
    drive_b = 4.0 * SQRT2 * epsilon_d_p * np.cos(2.0 * theta)

    du = np.empty_like(u)

    # state
    du[0] = v_a
    du[1] = v_b
    du[2] = -delta_ca * v_a - x_a + f_nl_a + 2.0 * epsilon * p_v_a
    du[3] = -delta_cb * v_b - 4.0 * x_b + f_nl_b - drive_b + 2.0 * epsilon * p_v_b
    du[4] = 1.0

    # costates
    # p_xdot = -∂H/∂x : includes BOTH p_v_a and p_v_b contributions
    du[5] = (p_v_a
             + 2.0 * beta_a_p * p_v_a * sin_s
             + 8.0 * beta_b_p * p_v_b * sin_s)

    du[6] = (4.0 * p_v_b
             + 2.0 * beta_a_p * p_v_a * sin_s
             + 8.0 * beta_b_p * p_v_b * sin_s)

    # p_vdot = -∂H/∂v
    du[7] = -p_x_a + delta_ca * p_v_a
    du[8] = -p_x_b + delta_cb * p_v_b

    return du

In [ ]:
tmax = 4/kappa_b_p
params = (0.0, 2.0 * kappa_b_p, eps_d_p, beta_a_p, beta_b_p, epsilon)
theta_0 = 0
a_R_0 = -0.18*1j*np.exp(-1j*theta_0)
x_a_0, v_a_0 = np.sqrt(2)*np.real(a_R_0), np.sqrt(2)*np.imag(a_R_0)
b_R_0 = -0.025*np.exp(-2j*theta_0)
x_b_0, v_b_0 = np.sqrt(2)*np.real(b_R_0), np.sqrt(2)*np.imag(b_R_0)*2

u_0 = np.concatenate(([x_a_0, x_b_0, v_a_0, v_b_0, theta_0], np.array([0,0,0,0,0])/1e3))  # length 9

sol = solve_ivp(
    fun=propagate_step,
    t_span=(0.0, tmax),
    y0=u_0,
    args=(params,),
    method="DOP853",
    t_eval = np.linspace(tmax-2*T,tmax-T,20),
    rtol=1e-9,
    atol=1e-12,
)

p_norm = [np.linalg.norm(((sol.y).T)[i][5:]) for i in range(len(sol.t))]
# Unpack solution (you already have this)
x_a, x_b, v_a, v_b, theta, p_x_a, p_x_b, p_v_a, p_v_b, p_theta = sol.y
x_a_s, x_b_s, v_a_s, v_b_s, theta_s = sol.y[:5]

check_H = False

if check_H :
    # Recompute the same pieces used in propagate_step_diag (vectorized)
    s = SQRT2 * (x_a + x_b)

    f_nl_a = SQRT2 * beta_a_p * (np.cos(s) - 1.0)
    f_nl_b = SQRT2 * (4.0 * beta_b_p) * (np.cos(s) - 1.0)
    drive_b = 4.0 * SQRT2 * eps_d_p * np.cos(2.0 * theta)

    delta_ca, delta_cb, epsilon_d_p, beta_a_p, beta_b_p, epsilon = params

    # Build the bracket B = δ_c V + K_nl(X) - f_ext(theta)
    # chosen to match your dv equations: dv = -B + 2ε P_V
    B_a = delta_ca * v_a + (x_a - f_nl_a)              # f_ext,a = 0
    B_b = delta_cb * v_b + (4.0 * x_b - f_nl_b) + drive_b  # note "+ drive_b" here

    # Hamiltonian along the trajectory
    H = (p_x_a * v_a + p_x_b * v_b) - (p_v_a * B_a + p_v_b * B_b) + p_theta + epsilon * (p_v_a**2 + p_v_b**2)
    H0 = H[0]
    dH = H - H0

    print("H0 =", H0)
    print("max |H - H0| =", np.max(np.abs(dH)))
    print("relative max drift =", np.max(np.abs(dH)) / (np.abs(H0) + 1e-30))

In [ ]:
theta_vals = theta_s%(2*np.pi)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10, 4))

ax_a, ax_b, ax_pnorm = axes

# complex amplitudes
a = (x_a + 1j*v_a)/np.sqrt(2)
b = (x_b + 1j*v_b/2)/np.sqrt(2)

t = sol.t
a_R = a*np.exp(1j*(t+theta_0))
b_R = b*np.exp(2j*(t+theta_0))

ax_a.plot(np.real(a_R[:]), np.imag(a_R[:]))
ax_b.plot(np.real(b_R[:]), np.imag(b_R[:]))
ax_pnorm.plot(p_norm[:])

# formatting
ax_a.set_title("Phase portrait: mode a")
ax_a.set_xlabel(r"$\Re(a_R)$")
ax_a.set_ylabel(r"$\Im(a_R)$")
#ax_a.set_xlim([-.5,.5])
#ax_a.set_ylim([-.5,.5])
ax_a.set_aspect('equal')
ax_a.grid(True)

ax_b.set_title("Phase portrait: mode b")
ax_b.set_xlabel(r"$\Re(b_R)$")
ax_b.set_ylabel(r"$\Im(b_R)$")
#ax_b.set_xlim([-.2,.5])
#ax_b.set_ylim([-.2,.2])
ax_b.set_aspect('equal')
ax_b.grid(True)

ax_pnorm.set_title("Distance to unstable orbits")
ax_pnorm.set_xlabel("Step")
ax_pnorm.set_ylabel(r"$|P|$")
ax_pnorm.set_yscale('log')
ax_pnorm.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
#n_theta = 10
n_q = 100
gamma = 1e-4
#theta_vals = np.linspace(0, T, n_theta, endpoint=False)

W_i = {}

for (i,theta_0) in enumerate(theta_vals):
    
    q_vals = sample_S3(gamma, n_q)

    # Base state at theta_0
    A_0_theta_0 = np.array([x_a_s[i],x_b_s[i],v_a_s[i],v_b_s[i]])
    q_cherry = np.zeros(4)#np.array([2.25533606e-01, 1.50988157e-02, 1.03395177e-01, 6.17822113e-02,])-A_0_theta_0#np.array([2.25522698e-01, 1.50051692e-02, 1.03365167e-01, 6.18296621e-02])

    # Jacobian blocks / (un)stable frame info at theta_0
    _, _, Zq_theta_0, Zp_theta_0 = Zq_Zp_from_theta(
        theta_0, T, orbit_state, J_extended, n_unstable=4
    )

    Zq_theta_0_inv = np.linalg.inv(Zq_theta_0)

    Wi_theta = []
    for q_val_i in q_vals:
        q = np.asarray(q_val_i).reshape(-1)+q_cherry  # ensure 1D vector

        # First block: perturbed state (requires A_0_theta_0 and q to be same shape)
        x = np.asarray(A_0_theta_0).reshape(-1) + q

        # Second block: mapped vector
        y = (Zp_theta_0 @ Zq_theta_0_inv @ q.reshape(-1, 1)).reshape(-1)

        Wi_theta.append(np.concatenate((x,[theta_0], y)))

    W_i[theta_0] = Wi_theta

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
ax_a, ax_b = axes
for theta_0 in theta_vals:
    x_a, v_a = W_i[theta_0][0][0:4:2]
    x_b, v_b = W_i[theta_0][0][1:4:2]
    a = (x_a + 1j*v_a)/np.sqrt(2)
    b = (x_b + 1j*v_b/2)/np.sqrt(2)

    a_R = a*np.exp(1j*theta_0)
    b_R = b*np.exp(2j*theta_0)

    ax_a.scatter(np.real(a_R), np.imag(a_R))
    ax_b.scatter(np.real(b_R), np.imag(b_R))
ax_a.set_title("Phase portrait: mode a")
ax_a.set_xlabel(r"$\Re(a_R)$")
ax_a.set_ylabel(r"$\Im(a_R)$")
ax_a.set_aspect('equal')
ax_a.grid(True)

ax_b.set_title("Phase portrait: mode b")
ax_b.set_xlabel(r"$\Re(b_R)$")
ax_b.set_ylabel(r"$\Im(b_R)$")
ax_b.set_aspect('equal')
ax_b.grid(True)
plt.show()

# MPEP

In [ ]:
def rk4_step(u, dt, rhs, params):
    """One RK4 step for u' = rhs(u). rhs must write into du and return du."""
    k1 = np.zeros_like(u)
    k2 = np.zeros_like(u)
    k3 = np.zeros_like(u)
    k4 = np.zeros_like(u)

    k1 = rhs(k1, u, params)
    k2 = rhs(k2, u + 0.5 * dt * k1, params)
    k3 = rhs(k3, u + 0.5 * dt * k2, params)
    k4 = rhs(k4, u + dt * k3, params)

    return u + (dt / 6.0) * (k1 + 2*k2 + 2*k3 + k4)

def integrate_with_distance_check(u0, rhs, params, dt=1e-3, t0=0.0, tmax=200.0, delta_Q=1e-8, delta_P=1e-10, check_every=1, store_every=10):
    """
    Integrate and check distance to unstable LC at each step (or every check_every steps).

    u0: initial 10D state
    rhs: function rhs(du, u, params) -> du (in-place)
    delta_Q: distance threshold to unstable LC (in weighted norm if weights provided)
    delta_P: threshold for momenta norm
    """

    u = np.array(u0, dtype=np.float64)
    t = float(t0)

    # storage
    times = []
    traj = []
    qnorms = []
    pnorms = []

    n_steps = int(np.ceil((tmax - t0) / dt))
    theta_0 = u0[4]

    for step in range(n_steps):
        #print (step/n_steps, end="\r")
        # check distance
        if step % check_every == 0:
            x_a,x_b,v_a,v_b = u[:4]#np.linalg.norm([u[0],u[2]])
            a = (x_a + 1j*v_a)/np.sqrt(2)
            b = (x_b + 1j*v_b/2)/np.sqrt(2)
            a_R = a*np.exp(1j*(t+theta_0))
            b_R = b*np.exp(2j*(t+theta_0))
            qnorm = min(np.linalg.norm(y_u - np.array([a_R, b_R])[:, None], axis= 0))

            pnorm = np.linalg.norm(u[5:9])
            # store occasionally
            if step % store_every == 0:
                #times.append(t)
                traj.append(u.copy())
                qnorms.append(qnorm)
                pnorms.append(pnorm)

            if qnorm > 1:
                #print("divergence")
                return {
                    "status": "divergence",
                    "t": t,
                    "u": u,
                    "traj": np.array(traj),
                    "qnorms": np.array(qnorms),
                    "pnorms":np.array(pnorms)
                }

            # stopping condition (near unstable + momenta small)
            if (qnorm < delta_Q) and (pnorm < delta_P):
                #print("success")
                #u = np.concatenate((u[:5],np.zeros(5)))                
                return {
                    "status": "hit_unstable",
                    "t": t,
                    "u": u,
                    "qnorm": qnorm,
                    "pnorm": pnorm,
                    "traj": np.array(traj),
                    "qnorms": np.array(qnorms),
                    "pnorms":np.array(pnorms)
                }

        # advance one step
        u = rk4_step(u, dt, rhs, params)
        t += dt
    
    # did not hit
    return {
        "status": "tmax_reached",
        "u": u,
        "traj": np.array(traj),
        "qnorms": np.array(qnorms),
        "pnorms": np.array(pnorms),
    }

In [ ]:
params = (0.0, 2.0 * kappa_b_p, eps_d_p, beta_a_p, beta_b_p, 10*epsilon)
tmax = 3.5/kappa_b_p
dt = 1e-2
store_every = 10
theta_0 = theta_vals[3]

u0 = np.array([2.25532723e-01, 1.50983730e-02, 1.03395054e-01, 6.17823072e-02,4.31120514e+00, 2.06516920e-05, 2.10811341e-05, 1.99459933e-05,2.01797539e-05,0])#np.concatenate((W_i[theta_vals[3]][59],[0]))#
res = integrate_with_distance_check(
            u0=u0,
            rhs=propagate_step,
            params=params,
            dt=dt,
            tmax=tmax,
            delta_Q=2.6e-3,
            delta_P=1e-3,
            check_every=1,
            store_every=store_every,
        )

In [ ]:
traj = [res["traj"][i][:4] for i in range(len(res["traj"]))]
traj = np.asarray(traj)
x_a, x_b, v_a, v_b = traj.T
min(res["qnorms"])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
ax_qnorm, ax_pnorm = axes
N_MAX = len(res["qnorms"])# np.argmin(res["qnorms"])
ax_qnorm.plot(res["qnorms"][:N_MAX])
ax_pnorm.plot(res["pnorms"][:N_MAX])


ax_qnorm.set_title("Distance to unstable orbits")
ax_qnorm.set_xlabel("Step")
ax_qnorm.set_ylabel(r"$|Q|$")
ax_qnorm.set_yscale('log')

ax_pnorm.set_title("Distance to unstable orbits")
ax_pnorm.set_xlabel("Step")
ax_pnorm.set_ylabel(r"$|P|$")
ax_pnorm.set_yscale('log')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
ax_a, ax_b = axes

# complex amplitudes
a = (x_a + 1j*v_a)/np.sqrt(2)
b = (x_b + 1j*v_b/2)/np.sqrt(2)

t = np.arange(0, tmax, store_every*dt)[:len(a)]
a_R = a*np.exp(1j*(t+theta_0))
b_R = b*np.exp(2j*(t+theta_0))

sc_a = ax_a.scatter(
    np.real(a_R),
    np.imag(a_R),
    s=0.01,
    c=t*kappa_b_p,
    cmap=Perso_colormap
)
ax_b.scatter(np.real(b_R[:]), np.imag(b_R[:]), s=.01, c=t*kappa_b_p, cmap=Perso_colormap)


ax_a.plot(np.real(a_R_u*np.exp(-0.05j)), np.imag(a_R_u*np.exp(-0.05j)), color='red')
ax_b.plot(np.real(b_R_u*np.exp(-0.1j)), np.imag(b_R_u*np.exp(-0.1j)), color='red')

#ax_a.scatter(np.real(a_R[N_MAX]), np.imag(a_R[N_MAX]), s=10, color="b")
#ax_b.scatter(np.real(b_R[N_MAX]), np.imag(b_R[N_MAX]), s=10, color="b")
# formatting
ax_a.set_title("Phase portrait: mode a")
ax_a.set_xlabel(r"$\Re(a_R)$")
ax_a.set_ylabel(r"$\Im(a_R)$")
#ax_a.set_xlim([-.05,.05])
#ax_a.set_ylim([-.05,.05])
ax_a.set_aspect('equal')
ax_a.grid(True)

ax_b.set_title("Phase portrait: mode b")
ax_b.set_xlabel(r"$\Re(b_R)$")
ax_b.set_ylabel(r"$\Im(b_R)$")
#ax_b.set_xlim([.2,.4])
#ax_b.set_ylim([-.1,.1])
ax_b.set_aspect('equal')
ax_b.grid(True)

fig.colorbar(sc_a, ax=ax_b, label="time "+r"$\kappa_b'\tau$")

plt.tight_layout()
plt.show()

In [ ]:
tmax = 2/kappa_b_p
dt = 1e-2
store_every = 100
params = (0, 2*kappa_b_p, eps_d_p, beta_a_p, beta_b_p, 10*epsilon)
res_list = []
for theta_0 in theta_vals[3:4]:
    print(theta_0)
    for i in range(n_q):
        print(i)
        res = integrate_with_distance_check(
            u0=W_i[theta_0][i],
            rhs=propagate_step_no_p_theta,
            params=params,
            dt=dt,
            tmax=tmax,
            delta_Q=1e-3,
            delta_P=1e-4,
            check_every=10,
            store_every=store_every,
        )
        res_list.append(res)

In [ ]:
W_i[theta_vals[3]][91]

In [ ]:
W_i[theta_vals[3]][65]

In [ ]:
W_i[theta_vals[3]][80]

In [ ]:
W_i[theta_vals[3]][3]

In [ ]:
a_R_list = []
b_R_list = []
dist_list = []
pnorms_list = []
t = np.arange(0, tmax, store_every*dt)
c=0 # counter
for res in res_list:
    #if res["status"] != "hit_unstable":
    if min(res["qnorms"]) <= 4e-1:
    
        theta_0 = res["traj"][0][4]
        traj = [res["traj"][i][:4] for i in range(len(res["traj"]))]
        traj = np.asarray(traj)
        x_a, x_b, v_a, v_b = traj.T
        t_i = t[:len(x_a)]
        a = (x_a + 1j*v_a)/np.sqrt(2)
        b = (x_b + 1j*v_b/2)/np.sqrt(2)
        a_R = a*np.exp(1j*(t_i+theta_0))
        b_R = b*np.exp(2j*(t_i+theta_0))
        a_R_list.append(a_R)
        b_R_list.append(b_R)

        dist_list.append(res["qnorms"])
        pnorms_list.append(res["pnorms"])
        print(c)
    c+=1

In [ ]:
W_i[theta_vals[3]][59]

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(10, 4))

ax_a, ax_b, ax_dist, ax_pnorm = axes
N_MAX = max([len(i) for i in a_R_list])#np.argmin(dist_list[0])#
N_MIN = N_MAX - 100
for i in range(len(a_R_list)):

    a_R = a_R_list[i]
    b_R = b_R_list[i]

    ax_a.plot(np.real(a_R[N_MIN:N_MAX]), np.imag(a_R[N_MIN:N_MAX]))
    ax_b.plot(np.real(b_R[N_MIN:N_MAX]), np.imag(b_R[N_MIN:N_MAX]))
    ax_dist.plot(dist_list[i][N_MIN:N_MAX])
    ax_pnorm.plot(pnorms_list[i][N_MIN:N_MAX])

ax_a.plot(np.real(a_R_u), np.imag(a_R_u), color='red')
ax_b.plot(np.real(b_R_u), np.imag(b_R_u), color='red')

# formatting
ax_a.set_title("Phase portrait: mode a")
ax_a.set_xlabel(r"$\Re(a_R)$")
ax_a.set_ylabel(r"$\Im(a_R)$")
ax_a.set_xlim([-.4,.4])
ax_a.set_ylim([-.4,.4]) 
ax_a.set_aspect('equal')
ax_a.grid(True)

ax_b.set_title("Phase portrait: mode b")
ax_b.set_xlabel(r"$\Re(b_R)$")
ax_b.set_ylabel(r"$\Im(b_R)$")
ax_b.set_xlim([-.2,.5])
ax_b.set_ylim([-.2,.2])
ax_b.set_aspect('equal')
ax_b.grid(True)

ax_dist.set_title("Distance to unstable orbits")
ax_dist.set_xlabel("Step")
ax_dist.set_ylabel("Dist")
ax_dist.set_yscale('log')
ax_dist.grid(True)

ax_pnorm.set_title("Norm of Ps")
ax_pnorm.set_xlabel("Step")
ax_pnorm.set_ylabel(r"$|P|$")
ax_pnorm.set_yscale('log')
ax_pnorm.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
W_i[theta_0][30]

# Shooting automatic

In [ ]:
angles = -0.05 + np.arange(11)/100

y_u_all = np.concatenate([
    np.vstack((
        a_R_u * np.exp(1j * angle_i / 2),
        b_R_u * np.exp(1j * angle_i)
    ))
    for angle_i in angles
], axis=1)

from scipy.spatial import cKDTree

# y_u_all: (2, M) complex
Y = np.column_stack([
    np.real(y_u_all[0]), np.imag(y_u_all[0]),
    np.real(y_u_all[1]), np.imag(y_u_all[1]),
]).astype(np.float64)          # shape (M, 4)

tree = cKDTree(Y)              # build once

In [ ]:
from joblib import Parallel , delayed

def integrate_qnorm(u0, rhs, params, dt=1e-3, t0=0.0, tmax=200.0, delta_Q=1e-8, delta_P=1e-10, check_every=1, store_every=10, record_pnorms=False):
    """
    Integrate and check distance to unstable LC at each step (or every check_every steps).

    u0: initial 10D state
    rhs: function rhs(du, u, params) -> du (in-place)
    delta_Q: distance threshold to unstable LC (in weighted norm if weights provided)
    delta_P: threshold for momenta norm
    """

    u = np.array(u0, dtype=np.float64)
    t = float(t0)

    # storage
    n_steps = int(np.ceil((tmax - t0) / dt))
    dists = np.ones(n_steps//store_every)
    if record_pnorms : pnorms = np.ones(n_steps//store_every)
    
    theta_0 = u0[4]
    inv_sqrt2 = 1.0 / np.sqrt(2.0)
    for step in range(n_steps):
        # check distance
        if step % check_every == 0:
            x_a,x_b,v_a,v_b = u[:4]#np.linalg.norm([u[0],u[2]])
            a = (x_a + 1j * v_a) * inv_sqrt2
            b = (x_b + 1j * v_b * 0.5) * inv_sqrt2
            a_R = a*np.exp(1j*(t+theta_0))
            b_R = b*np.exp(2j*(t+theta_0))
            q = np.array([a_R.real, a_R.imag, b_R.real, b_R.imag], dtype=np.float64)
            dist, _ = tree.query(q, k=1)   # Euclidean distance
            pnorm = np.linalg.norm(u[5:])

            # store occasionally
            if step % store_every == 0:
                dists[step // store_every -1]=dist
                if record_pnorms : pnorms[step // store_every -1]=pnorm

            if np.imag(a_R) > 0.2:
                break

        # advance one step
        u = rk4_step(u, dt, rhs, params)
        t += dt
    
    if record_pnorms : return {"u0": u0,"dists": np.array(dists),"pnorms": np.array(pnorms)}
    return {"u0": u0,"dists": np.array(dists)}

def find_n_optimal_paths(n_keep, u0_list, rhs, params, dt, tmax, delta_Q = 1e-3, delta_P=1e-4, check_every=10, store_every=100):
    res_list = Parallel(n_jobs=-1, prefer="processes")(
        delayed(integrate_qnorm)(
            u0=u0,
            rhs=rhs,
            params=params,
            dt=dt,
            tmax=tmax,
            delta_Q=delta_Q,
            delta_P=delta_P,
            check_every=check_every,
            store_every=store_every,
        )
        for u0 in u0_list
    )
    # Compute each run's minimal qnorm
    min_dist = np.array([np.min(res["dists"]) for res in res_list], dtype=float)

    # Indices of the n_keep smallest minima
    n_keep = min(int(n_keep), len(min_dist))
    idx_keep = np.argpartition(min_dist, n_keep - 1)[:n_keep]
    idx_keep = idx_keep[np.argsort(min_dist[idx_keep])]  # sorted best -> worst

    # Return the kept initial conditions and their scores (and optionally full results)
    u0_keep = [res_list[i]["u0"] for i in idx_keep]
    dist_keep = min_dist[idx_keep]

    return u0_keep, dist_keep

def sample_theta(theta_0, A_0_theta_0, n_q, gamma, q_cherry = np.zeros(4)):
    q_vals = sample_S3(gamma, n_q)

    # Jacobian blocks / (un)stable frame info at theta_0
    _, _, Zq_theta_0, Zp_theta_0 = Zq_Zp_from_theta(
        theta_0, T, orbit_state, J_extended, n_unstable=4
    )

    Zq_theta_0_inv = np.linalg.inv(Zq_theta_0)

    Wi_theta = []
    for q_val_i in q_vals:
        q = np.asarray(q_val_i).reshape(-1)+q_cherry  # ensure 1D vector

        # First block: perturbed state (requires A_0_theta_0 and q to be same shape)
        x = np.asarray(A_0_theta_0).reshape(-1) + q

        # Second block: mapped vector
        y = (Zp_theta_0 @ Zq_theta_0_inv @ q.reshape(-1, 1)).reshape(-1)

        Wi_theta.append(np.concatenate((x,[theta_0], y)))

    return Wi_theta

In [ ]:
tmax = 3 / kappa_b_p
dt = 1e-2
store_every = 100
rhs = propagate_step_no_p_theta
params = (0, 2*kappa_b_p, eps_d_p, beta_a_p, beta_b_p, 10*epsilon)
idx_theta_0 = 3
theta_0 = theta_vals[idx_theta_0]
A_0_theta_0 = np.array([x_a_s[idx_theta_0],x_b_s[idx_theta_0],v_a_s[idx_theta_0],v_b_s[idx_theta_0]])
n_q = 100
gamma = 1e-4
Wi_theta = sample_theta(theta_0, A_0_theta_0, n_q, gamma, q_cherry = np.zeros(4))
n_q = 20
n_keep = 4
for i in range(4):
    u0_keep, dist_opt = find_n_optimal_paths(n_keep, Wi_theta, rhs, params, dt, tmax, delta_Q = 1e-3, delta_P=1e-4, check_every=10, store_every=100)
    gamma /= 4
    Wi_theta_list = []
    for j in range(n_keep):
        u_0_opt = u0_keep[j]
        Wi_theta_j = sample_theta(theta_0,A_0_theta_0,n_q,gamma,q_cherry=u_0_opt[:4] - A_0_theta_0)
        Wi_theta_list.append(Wi_theta_j)
    Wi_theta = np.concatenate(Wi_theta_list, axis=0)
    print(dist_opt)
u0_keep, dist_opt = find_n_optimal_paths(1, Wi_theta, rhs, params, dt, tmax, delta_Q = 1e-3, delta_P=1e-4, check_every=10, store_every=100)
print(dist_opt)

In [ ]:
u = np.concatenate((u0_keep[0],[0]))
t0 = 0
t = t0
n_steps = int(np.ceil((tmax - t0) / dt))
traj=[]
for step in range(n_steps):
    # store occasionally
    if step % store_every == 0:
        traj.append(u)
    # advance one step
    u = rk4_step(u, dt, propagate_step, params)
    t += dt
traj = np.array(traj)
pnorms = [np.linalg.norm(point[5:-1]) for point in traj]
dists = []
for point in traj:
    x_a,x_b,v_a,v_b = point[:4]
    a = (x_a + 1j * v_a) / np.sqrt(2)
    b = (x_b + 1j * v_b * 0.5) / np.sqrt(2)
    a_R = a*np.exp(1j*(t+theta_0))
    b_R = b*np.exp(2j*(t+theta_0))
    q = np.array([a_R.real, a_R.imag, b_R.real, b_R.imag], dtype=np.float64)
    dist, _ = tree.query(q, k=1)   # Euclidean distance
    dists.append(dist)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
ax_qnorm, ax_pnorm = axes
ax_qnorm.plot(dists[:])
ax_pnorm.plot(pnorms[:])


ax_qnorm.set_title("Distance to unstable orbits")
ax_qnorm.set_xlabel("Step")
ax_qnorm.set_ylabel(r"$|Q|$")
ax_qnorm.set_yscale('log')

ax_pnorm.set_title("Distance to unstable orbits")
ax_pnorm.set_xlabel("Step")
ax_pnorm.set_ylabel(r"$|P|$")
ax_pnorm.set_yscale('log')

In [ ]:
N_MAX = np.argmin(dists)
x_a, x_b, v_a, v_b, theta, p_x_a, p_x_b, p_v_a, p_v_b, p_theta = traj[:N_MAX].T
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
ax_a, ax_b = axes

# complex amplitudes
a = (x_a + 1j*v_a)/np.sqrt(2)
b = (x_b + 1j*v_b/2)/np.sqrt(2)

t = np.arange(0, tmax, store_every*dt)[:len(a)]
a_R = a*np.exp(1j*(t+theta_0))
b_R = b*np.exp(2j*(t+theta_0))

sc_a = ax_a.scatter(
    np.real(a_R),
    np.imag(a_R),
    s=0.1,
    c=t*kappa_b_p,
    cmap=Perso_colormap
)
ax_b.scatter(np.real(b_R[:]), np.imag(b_R[:]), s=.1, c=t*kappa_b_p, cmap=Perso_colormap)


#ax_a.plot(np.real(a_R_u*np.exp(-0.05j)), np.imag(a_R_u*np.exp(-0.05j)), color='red')
#ax_b.plot(np.real(b_R_u*np.exp(-0.1j)), np.imag(b_R_u*np.exp(-0.1j)), color='red')

#ax_a.scatter(np.real(a_R[N_MAX]), np.imag(a_R[N_MAX]), s=10, color="b")
#ax_b.scatter(np.real(b_R[N_MAX]), np.imag(b_R[N_MAX]), s=10, color="b")
# formatting
ax_a.set_title("Phase portrait: mode a")
ax_a.set_xlabel(r"$\Re(a_R)$")
ax_a.set_ylabel(r"$\Im(a_R)$")
ax_a.set_xlim([-.5,.5])
ax_a.set_ylim([-.5,.5])
ax_a.set_aspect('equal')
ax_a.grid(True)

ax_b.set_title("Phase portrait: mode b")
ax_b.set_xlabel(r"$\Re(b_R)$")
ax_b.set_ylabel(r"$\Im(b_R)$")
ax_b.set_xlim([-.5,.5])
ax_b.set_ylim([-.5,.5])
ax_b.set_aspect('equal')
ax_b.grid(True)

fig.colorbar(sc_a, ax=ax_b, label="time "+r"$\kappa_b'\tau$")

plt.tight_layout()
plt.show()

# Computation of the action along the path

In [ ]:
s = SQRT2 * (x_a + x_b)

f_nl_a = SQRT2 * beta_a_p * (np.cos(s) - 1.0)
f_nl_b = SQRT2 * (4.0 * beta_b_p) * (np.cos(s) - 1.0)
drive_b = 4.0 * SQRT2 * eps_d_p * np.cos(2.0 * theta)

delta_ca, delta_cb, epsilon_d_p, beta_a_p, beta_b_p, epsilon = params

# Build the bracket B = δ_c V + K_nl(X) - f_ext(theta)
# chosen to match your dv equations: dv = -B + 2ε P_V
B_a = delta_ca * v_a + (x_a - f_nl_a)              # f_ext,a = 0
B_b = delta_cb * v_b + (4.0 * x_b - f_nl_b) + drive_b  # note "+ drive_b" here

# Hamiltonian along the trajectory
H = (p_x_a * v_a + p_x_b * v_b) - (p_v_a * B_a + p_v_b * B_b) + p_theta + epsilon * (p_v_a**2 + p_v_b**2)

dS = p_x_a[1:]*np.diff(x_a) + p_x_b[1:]*np.diff(x_b) + p_v_a[1:]*np.diff(v_a) + p_v_b[1:]*np.diff(v_b) + p_theta[1:]*np.diff(theta) - H[1:]*dt
S = -np.sum(dS)
np.exp(S)